# Phase 2: Enhanced Hybrid GraphRAG Retrieval (Weaviate + Neo4j)

**7-Step Pipeline** (Aligned with Full Ontology & Weaviate):
1. Keywords VDB Hybrid Search → Content clues
2. LLM → High-Level (HL) + Low-Level (LL) keywords
3. Entity VDB Hybrid Search (LL) → Seed Entities
4. Neo4j Subgraph Traversal (2-hop) + Co-authorship network
5. Relationship VDB Hybrid Search (HL) → Global context
6. Chunk VDB Hybrid Search → Text similarity
7. Context fusion + LLM generation (with paper URLs)

In [28]:
# Cell 1: Setup & Verify Connections
import os, json, requests, warnings, time
warnings.filterwarnings('ignore')

from neo4j import GraphDatabase
import weaviate
from weaviate.classes.query import MetadataQuery
from dotenv import load_dotenv

# Adjust path to .env if needed
load_dotenv('../../.env') 
NEO4J_URI   = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER  = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASS  = os.getenv('NEO4J_PASS', 'rizkyyk123')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
WEAVIATE_PORT = 8081

wv_client = weaviate.connect_to_local(port=WEAVIATE_PORT)
print(f"Weaviate Ready: {wv_client.is_ready()}")

# Verify all 4 VDB collections exist
for name in ['EntityEmbedding', 'RelationshipEmbedding', 'ContentKeyword', 'PaperChunk']:
    try:
        col = wv_client.collections.get(name)
        cnt = col.aggregate.over_all().total_count
        print(f'{name}: {cnt} entries')
    except Exception as e:
        print(f'{name}: ERROR - {e}')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with driver.session() as s:
    print(f"Neo4j: {s.run('MATCH (n) RETURN count(n) as c').single()['c']} nodes")

print(f'OpenRouter: {"OK" if OPENROUTER_API_KEY else "NOT SET!"}')

Weaviate Ready: True
EntityEmbedding: 192 entries
RelationshipEmbedding: 35 entries
ContentKeyword: 50 entries
PaperChunk: 50 entries
Neo4j: 330 nodes
OpenRouter: OK


In [29]:
# Cell 2: LLM Helper
def call_openrouter(prompt, json_mode=True, system_prompt=None):
    url = 'https://openrouter.ai/api/v1/chat/completions'
    headers = {'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'}
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    payload = {
        'model': 'google/gemini-2.0-flash-001',
        'messages': messages,
        'temperature': 0.3
    }
    if json_mode:
        payload['response_format'] = {'type': 'json_object'}
    try:
        res = requests.post(url, headers=headers, json=payload, timeout=30)
        content = res.json()['choices'][0]['message']['content']
        return json.loads(content) if json_mode else content
    except Exception as e:
        print(f'  LLM error: {e}')
        return {} if json_mode else ''

print('LLM helper ready.')

LLM helper ready.


In [30]:
# Cell 3: Step 1 - Clue-Guided Keyword Extraction
def step1_extract_keywords(query):
    # 1a. Hybrid search Keywords VDB for content clues
    col = wv_client.collections.get("ContentKeyword")
    results = col.query.hybrid(query=query, alpha=0.5, limit=5)
    clues = [obj.properties.get('keywords', '') for obj in results.objects]
    clue_text = '; '.join(clues[:5])

    # 1b. LLM extracts HL/LL keywords using clues
    prompt = f"""Anda keyword extractor untuk Knowledge Graph akademik.
Dari pertanyaan DAN content keywords (clue), ekstrak:
1. high_level_keywords: tema/topik besar (untuk relasi global)
2. low_level_keywords: entitas spesifik (untuk seed graph search)
3. vector_query: ringkasan kalimat untuk pencarian similarity

Content Keywords (Clues): {clue_text}
Pertanyaan: "{query}"

Output JSON:
{{"high_level_keywords": ["..."], "low_level_keywords": ["..."], "vector_query": "..."}}"""
    result = call_openrouter(prompt)
    hl = result.get('high_level_keywords', [query])
    ll = result.get('low_level_keywords', [query])
    vq = result.get('vector_query', query)
    return {'hl': hl, 'll': ll, 'vq': vq}

print("Step 1 Ready.")

Step 1 Ready.


In [31]:
# Cell 4: Step 2 - Entity VDB Hybrid Search
def step2_entity_search(ll_keywords, top_k=10):
    col = wv_client.collections.get("EntityEmbedding")
    all_results = []
    seen = set()
    
    for kw in ll_keywords[:5]:
        results = col.query.hybrid(query=kw, alpha=0.5, limit=top_k, return_metadata=MetadataQuery(score=True))
        for obj in results.objects:
            nid = obj.properties.get('nodeId', '')
            if nid and nid not in seen:
                seen.add(nid)
                all_results.append({
                    'node_id': nid, 
                    'name': obj.properties.get('entityName', ''),
                    'type': obj.properties.get('entityType', ''),
                    'score': obj.metadata.score
                })
    all_results.sort(key=lambda x: x['score'], reverse=True)
    return all_results[:top_k]

print("Step 2 Ready.")

Step 2 Ready.


In [32]:
# Cell 5: Step 3 - Subgraph Traversal (Neo4j 2-hop)
def step3_subgraph_traversal(entity_matches, hops=2):
    context_parts = []
    with driver.session() as session:
        for ent in entity_matches[:5]:
            nid = ent['node_id']
            result = session.run(f"""
                MATCH (start {{node_id: $nid}})
                OPTIONAL MATCH path = (start)-[r*1..{hops}]-(neighbor)
                WITH start, neighbor, r
                WHERE neighbor IS NOT NULL
                RETURN start.node_id as src, labels(start)[0] as slbl,
                       coalesce(start.name, start.title) as sname,
                       neighbor.node_id as tgt, labels(neighbor)[0] as tlbl,
                       coalesce(neighbor.name, neighbor.title) as tname,
                       type(last(r)) as rel
                LIMIT 20
            """, nid=nid)
            for rec in result:
                line = f"({rec['slbl']}: {rec['sname']}) -[{rec['rel']}]-> ({rec['tlbl']}: {rec['tname']})"
                context_parts.append(line)
    return '\n'.join(list(set(context_parts))[:25])

print("Step 3 Ready.")

Step 3 Ready.


In [33]:
# Cell 6: Step 4 - Relationship VDB Hybrid Search
def step4_relationship_search(hl_keywords, top_k=5):
    col = wv_client.collections.get("RelationshipEmbedding")
    all_results = []
    for kw in hl_keywords[:3]:
        results = col.query.hybrid(query=kw, alpha=0.5, limit=top_k)
        for obj in results.objects:
            all_results.append(obj.properties.get('description', ''))
    return '\n'.join(list(set(all_results))[:top_k])

print("Step 4 Ready.")

Step 4 Ready.


In [34]:
# Cell 7: Step 5 - Chunk VDB Hybrid Search
def step5_chunk_search(vq, top_k=5):
    col = wv_client.collections.get("PaperChunk")
    results = col.query.hybrid(query=vq, alpha=0.5, limit=top_k)
    chunks = []
    for obj in results.objects:
        chunks.append({
            'title': obj.properties.get('title', ''),
            'text': obj.properties.get('content', '')[:500],
            'url': obj.properties.get('paperUrl', '')
        })
    return chunks

print("Step 5 Ready.")

Step 5 Ready.


In [35]:
# Cell 8: Step 6-7 - Context Fusion + Generation
def step6_generate_answer(query, subgraph_ctx, rel_ctx, chunks):
    chunk_text = ''
    for i, c in enumerate(chunks[:5], 1):
        url_str = f"\n   URL: {c['url']}" if c['url'] else "\n   URL: (Tidak ada link di dataset)"
        chunk_text += f"\n[Paper {i}] {c['title']}{url_str}\n   {c['text']}\n"
        
    system_prompt = """Anda asisten riset akademik UNESA Infokom (Consensus-style).
SELALU cantumkan nama dosen, judul paper, dan URL jika tersedia. Jika URL tidak ada, cantumkan (Tidak ada link di dataset).
Jawab dalam Bahasa Indonesia yang terstruktur dengan bullet points."""

    prompt = f"""Pernyataan: {query}

=== KONTEKS GRAPH ===
{subgraph_ctx}

=== KONTEKS RELASI GLOBAL ===
{rel_ctx}

=== KONTEKS CHUNK PAPER ===
{chunk_text}"""
    
    return call_openrouter(prompt, json_mode=False, system_prompt=system_prompt)

print("Generator ready.")

Generator ready.


In [36]:
hybrid_graphrag_query("Siapa dosen yang meneliti tentang deep learning atau machine learning?")
hybrid_graphrag_query("Berikan rekomendasi dosen jika saya interest di bidang IoT dan data mining, lalu carikan saya paper dosen beserta link papernya!")
hybrid_graphrag_query("Apa saja metode eksperimen atau model komputasi yang paling sering dipakai di Fakultas Teknik berdasarkan paper mereka?")
hybrid_graphrag_query("Adakah kolaborasi pembuatan sistem informasi atau aplikasi mobile di prodi S1 Pendidikan Teknologi Informasi?")
hybrid_graphrag_query("Sebutkan problem utama atau metrik pengujian yang paling populer dikaji oleh dosen Informatika tahun 2022-2023!")



PERTANYAAN: Siapa dosen yang meneliti tentang deep learning atau machine learning?



JAWABAN:
 Berikut adalah daftar dosen yang meneliti tentang deep learning atau machine learning, berdasarkan informasi yang tersedia:

*   **Paper 1**: Implementasi Algoritma Machine Learning Untuk Pengelolaan Laporan Pada Website Dinas Penanaman Modal Dan Pelayanan Terpadu Satu Pintu (Dpmptsp) Kota Surabaya.
    *   URL: [https://scholar.google.com/citations?view_op=view_citation&hl=en&user=N6RJilIAAAAJ&pagesize=100&sortby=pubdate&citation_for_view=N6RJilIAAAAJ:D03iK_w7-QYC](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=N6RJilIAAAAJ&pagesize=100&sortby=pubdate&citation_for_view=N6RJilIAAAAJ:D03iK_w7-QYC)
*   **Paper 2**: Klasifikasi Gambar Asli Dan Manipulasi Menggunakan Error Level Analysis (Ela) Sebagai Proses Komputasi Metode Convolutional Neural Network (Cnn).
    *   URL: [https://ejournal.unesa.ac.id/index.php/jinacs/article/download/36665/32514](https://ejournal.unesa.ac.id/index.php/jinacs/article/download/36665/32514)

PERTANYAAN: Berikan rekomendasi 